# 05 · Weekly Boss Update
Auto-generates a concise weekly summary for leadership covering team activity, use case gaps, key wins, and AI-extracted themes from customer conversations.

In [ ]:
import datetime, json
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.gridspec as gs
import pandas as pd
from IPython.display import display, HTML
plt.switch_backend("agg")

SF_BLUE="#29B5E8"; SF_TEAL="#00A9CE"; SF_GREEN="#36B37E"
SF_ORANGE="#FF8B00"; SF_DARK="#0A2859"; SF_GRAY="#D0D0D0"; BG="#F9FAFB"

def clean(ax):
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_visible(False)

def fmt_eacv(v):
    if not v or (isinstance(v,float) and str(v)=="nan"): return "$0"
    v=float(v)
    return f"${v/1e6:.1f}M" if v>=1e6 else f"${v/1e3:.0f}K" if v>=1e3 else f"${v:.0f}"

def html_table(df, row_color_fn=None):
    if hasattr(df, 'to_pandas'): df = df.to_pandas()
    html = "<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>"
    html += "<tr style='background:#0A2859;color:white'>" + "".join("<th style='padding:8px 12px;text-align:left'>" + str(col) + "</th>" for col in df.columns) + "</tr>"
    for i, (_, r) in enumerate(df.iterrows()):
        bg = row_color_fn(r) if row_color_fn else ("#f8f9fa" if i%2==0 else "white")
        html += "<tr style='background:" + bg + "'>" + "".join("<td style='padding:7px 12px'>" + str(v if v is not None else '') + "</td>" for v in r) + "</tr>"
    html += "</table>"
    display(HTML(html))

TEAM = {
    "Jim Lebonitte": "005VI00000Z0y6bYAB",
    "Michael Hamilton": "005VI00000ExC3VYAV",
    "Patrick Sheehan": "0053r00000BblkZAAR",
    "Phani Alapaty": "005VI00000HajknYAB",
    "Steve Mitchener": "0050Z000009IrKRQA0",
    "Zaki Bajwa": "005VI00000XibQfYAJ"
}
TEAM_IDS   = list(TEAM.values())
team_ids_str = "', '".join(TEAM_IDS)
ACT_TABLE  = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"

fy_start   = datetime.date(2026, 2, 1)
q2_start   = datetime.date(2026, 5, 1)
today      = datetime.date.today()
fy_start_str = str(fy_start)
q2_start_str = str(q2_start)
today_str    = str(today)
print("Setup complete ✓")

# ─── FILTERS ─────────────────────────────────────────────
week_override = None  # None = last full Mon-Sun week, or datetime.date(2026,6,9)
# ─────────────────────────────────────────────────────────
if week_override:
    week_start = week_override - datetime.timedelta(days=week_override.weekday())
else:
    days_since_mon = today.weekday()
    week_start = today - datetime.timedelta(days=days_since_mon + 7)
week_end = week_start + datetime.timedelta(days=6)
ws, we = str(week_start), str(week_end)
print(f"Reporting week: {ws} → {we}")


## This Week at a Glance

In [ ]:
%%sql -r this_week
SELECT
    COUNT(*)                                                    AS total_meetings,
    COUNT(DISTINCT SF_ACCOUNT_ID)                               AS unique_accounts,
    COUNT(DISTINCT MEETING_SE_NAME)                             AS active_specialists,
    SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))                    AS setsail_matched,
    SUM(IFF(SF_ACTIVITY_ID IS NULL,1,0))                        AS backlog,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))                   AS with_use_case,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='No' AND SF_ACCOUNT_ID IS NOT NULL,1,0)) AS uc_gap,
    SUM(IFF(SUMMARY IS NOT NULL,1,0))                           AS gong_summaries
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'

In [ ]:
%%sql -r last_week
SELECT
    COUNT(*) AS total_meetings,
    COUNT(DISTINCT SF_ACCOUNT_ID) AS unique_accounts
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN DATEADD('day',-7,'{{ws}}') AND DATEADD('day',-1,'{{ws}}')

In [ ]:
last_week = last_week.to_pandas() if hasattr(last_week, "to_pandas") else last_week
this_week = this_week.to_pandas() if hasattr(this_week, "to_pandas") else this_week
tw = this_week.iloc[0]; lw = last_week.iloc[0]
mtgs = int(tw["TOTAL_MEETINGS"] or 0); lmtgs = int(lw["TOTAL_MEETINGS"] or 0)
accts = int(tw["UNIQUE_ACCOUNTS"] or 0); laccts = int(lw["UNIQUE_ACCOUNTS"] or 0)
delta_m = mtgs - lmtgs; delta_a = accts - laccts
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax in axes: ax.set_facecolor(BG); ax.axis("off")
kpis = [
    (mtgs, f"{'▲' if delta_m>=0 else '▼'}{abs(delta_m)} vs last wk", "Meetings\nThis Week", SF_BLUE),
    (accts, f"{'▲' if delta_a>=0 else '▼'}{abs(delta_a)} vs last wk", "Unique\nAccounts", SF_TEAL),
    (int(tw["UC_GAP"] or 0), "need use case created", "Use Case\nGap", SF_ORANGE),
    (int(tw["BACKLOG"] or 0), "not matched by Setsail", "Setsail\nUnmatched", "#E74C3C"),
]
for ax, (val, sub, title, color) in zip(axes, kpis):
    ax.text(0.5, 0.65, str(val), ha="center", va="center", fontsize=36, fontweight="bold", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.35, sub, ha="center", va="center", fontsize=9, color=SF_GRAY, transform=ax.transAxes)
    ax.text(0.5, 0.1, title, ha="center", va="center", fontsize=11, color=SF_DARK, fontweight="bold", transform=ax.transAxes)
plt.suptitle(f"Week of {ws}", color=SF_DARK, fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## By Specialist

In [ ]:
%%sql -r by_member
SELECT
    COALESCE(MEETING_SE_NAME,'Unknown')  AS specialist,
    COUNT(*)                              AS meetings,
    COUNT(DISTINCT SF_ACCOUNT_ID)         AS accounts,
    SUM(IFF(SF_ACTIVITY_ID IS NULL,1,0))  AS unlogged,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='No' AND SF_ACCOUNT_ID IS NOT NULL,1,0)) AS uc_gap
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'
GROUP BY 1 ORDER BY 2 DESC

In [ ]:
by_member = by_member.to_pandas() if hasattr(by_member, "to_pandas") else by_member
html_table(by_member)

## Key Customer Meetings This Week

In [ ]:
%%sql -r key_meetings
SELECT
    MEETING_DATE, MEETING_SE_NAME AS specialist,
    CUSTOMER, MEETING_TITLE,
    USE_CASE_NAME, OPPORTUNITY_NAME,
    LEFT(SUMMARY, 300) AS summary_preview
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'
  AND CUSTOMER IS NOT NULL AND CUSTOMER != ''
ORDER BY IFF(SUMMARY IS NOT NULL,0,1), MEETING_DATE

In [ ]:
key_meetings = key_meetings.to_pandas() if hasattr(key_meetings, "to_pandas") else key_meetings
html_table(key_meetings)

## AI — Weekly Themes & Highlights

In [ ]:
%%sql -r week_summaries
SELECT CUSTOMER, MEETING_SE_NAME, SUMMARY, NEXT_STEPS
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'
  AND SUMMARY IS NOT NULL AND TRIM(SUMMARY) != ''

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
week_summaries = week_summaries.to_pandas() if hasattr(week_summaries, "to_pandas") else week_summaries
texts = "\n---\n".join(
    f"[{r['MEETING_SE_NAME']} @ {r['CUSTOMER']}] {r['SUMMARY']}"
    for _, r in week_summaries.iterrows() if r['SUMMARY']
)
if texts.strip():
    prompt = (
        f"You are preparing a weekly update for a manager of a Snowflake specialist team (week of {ws}).\n"
        "Based on these customer meeting summaries, write a concise weekly update covering:\n"
        "1. Top 3 customer themes/conversations this week\n"
        "2. Any urgent customer needs or blockers mentioned\n"
        "3. Competitive mentions or deal risk signals\n"
        "4. 2-3 recommended follow-up actions for the team\n"
        "Keep it to 200-250 words, executive-ready tone.\n\nMeeting summaries:\n" + texts[:5000]
    )
    result = session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-7b', ?)", [prompt]).collect()[0][0]
    print(f"=== Weekly Narrative ({ws} → {we}) ===\n")
    print(result)
else:
    print("No Gong summaries this week.")

## Accounts Going Cold (>30 Days No Touch)

In [ ]:
%%sql -r cold
SELECT CUSTOMER, SF_ACCOUNT_ID, MEETING_SE_NAME AS last_specialist,
       MAX(MEETING_DATE) AS last_touch,
       DATEDIFF('day', MAX(MEETING_DATE), CURRENT_DATE()) AS days_since
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE CUSTOMER IS NOT NULL
GROUP BY 1,2,3
HAVING days_since > 30
ORDER BY 5 DESC LIMIT 15

In [ ]:
cold = cold.to_pandas() if hasattr(cold, "to_pandas") else cold
html_table(cold)